# Vision AI — Colab Runner

**What this project is:** a real-time accessibility vision system for detecting everyday objects and Indian rupee notes, with voice announcements.

| Module | Role |
|---|---|
| `object_detection.py` | YOLOv11 object detection + NMS + temporal tracking |
| `currency_detection.py` | Indian notes (₹10–₹500) via custom YOLO or color fallback |
| `preprocessing.py` | Brightness / CLAHE / denoise pipeline |
| `voice_engine.py` | Queued TTS with cooldowns (print/gTTS on Colab) |
| `camera.py` + `r.py` | Threaded webcam + Flask UI (desktop; Colab uses Gradio) |

**Colab adaptations**
- No local webcam / Flask desktop UI → **Gradio** interactive app
- `pyttsx3` unreliable in Colab → **text log + optional gTTS audio**
- Upload `best.pt` for currency, or use color/COCO fallback

**Runtime:** Runtime → Change runtime type → **GPU (T4)** recommended.

## 1. Install dependencies

In [ ]:
!pip -q install ultralytics opencv-python-headless pillow gradio gTTS IPython

import torch, cv2, ultralytics
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenCV:", cv2.__version__)
print("Ultralytics:", ultralytics.__version__)

## 2. Project folders + model weights

- `yolo11n.pt` downloads automatically on first object-detection run.
- Optional: upload your trained `best.pt` (currency) into `weights/`.

In [ ]:
import os
from pathlib import Path

ROOT = Path("/content/vision_ai")
WEIGHTS = ROOT / "weights"
LOGS = ROOT / "logs"
SAMPLES = ROOT / "samples"

for d in (ROOT, WEIGHTS, LOGS, SAMPLES):
    d.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)
print("Working dir:", os.getcwd())

# Optional: upload best.pt from your machine (currency model)
UPLOAD_CURRENCY_MODEL = False  # set True, then run this cell

if UPLOAD_CURRENCY_MODEL:
    from google.colab import files
    print("Upload best.pt …")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = WEIGHTS / "best.pt"
        with open(dest, "wb") as f:
            f.write(data)
        print("Saved", dest, f"({len(data)} bytes)")

print("Weights present:", list(WEIGHTS.glob("*")))


## 3. Configuration

In [ ]:
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, Any
import json

@dataclass
class CameraConfig:
    camera_id: int = 0
    fps: int = 30
    resize_scale: float = 0.75
    frame_buffer_size: int = 5
    skip_frames: int = 2

@dataclass
class ObjectDetectionConfig:
    model_path: str = "yolo11n.pt"  # auto-download
    conf_threshold: float = 0.5
    nms_threshold: float = 0.45
    apply_tracking: bool = True
    apply_filtering: bool = True
    min_box_size: int = 10
    min_area: int = 400

@dataclass
class CurrencyDetectionConfig:
    model_path: str = "weights/best.pt"
    use_custom_model: bool = True
    conf_threshold: float = 0.6
    apply_low_light_enhancement: bool = True
    detection_cooldown: float = 4.0

@dataclass
class PreprocessingConfig:
    normalize_brightness: bool = True
    enhance_contrast: bool = True
    denoise: bool = True
    sharpen: bool = False
    denoise_strength: int = 8
    clahe_clip: float = 2.0
    target_brightness: int = 100

@dataclass
class VoiceConfig:
    enabled: bool = True
    rate: int = 160
    volume: float = 1.0
    global_cooldown: float = 1.5
    object_cooldown_duration: float = 3.0
    use_gtts: bool = True  # Colab-friendly TTS

@dataclass
class SystemConfig:
    mode: Optional[str] = None
    use_gpu: bool = True
    camera: CameraConfig = field(default_factory=CameraConfig)
    object_detection: ObjectDetectionConfig = field(default_factory=ObjectDetectionConfig)
    currency_detection: CurrencyDetectionConfig = field(default_factory=CurrencyDetectionConfig)
    preprocessing: PreprocessingConfig = field(default_factory=PreprocessingConfig)
    voice: VoiceConfig = field(default_factory=VoiceConfig)

CFG = SystemConfig()
print(json.dumps({
    "object": asdict(CFG.object_detection),
    "currency": asdict(CFG.currency_detection),
    "preprocessing": asdict(CFG.preprocessing),
}, indent=2))

## 4. Core modules (self-contained)

In [ ]:
import time
import queue
import threading
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
from IPython.display import Audio, display

# -------------------- Preprocessing --------------------
class FramePreprocessor:
    def __init__(self, enable_adaptive_histogram=True):
        self.enable_adaptive_histogram = enable_adaptive_histogram
        self.prev_brightness = 0

    def normalize_brightness(self, frame, target_brightness=100):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        current = float(np.mean(lab[:, :, 0]))
        self.prev_brightness = 0.8 * self.prev_brightness + 0.2 * current
        if self.prev_brightness < target_brightness - 10:
            lab[:, :, 0] = cv2.add(lab[:, :, 0], 20)
        elif self.prev_brightness > target_brightness + 10:
            lab[:, :, 0] = cv2.subtract(lab[:, :, 0], 15)
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    def enhance_contrast(self, frame, clip_limit=2.0):
        if not self.enable_adaptive_histogram:
            return frame
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    def denoise(self, frame, strength=8):
        return cv2.bilateralFilter(frame, 9, strength, strength)

    def sharpen(self, frame, strength=1.0):
        kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]], dtype=np.float32)
        sharpened = cv2.filter2D(frame, -1, kernel)
        return cv2.addWeighted(frame, 1.0 - strength / 10, sharpened, strength / 10, 0)

    def preprocess_for_detection(self, frame, config=None):
        config = config or asdict(CFG.preprocessing)
        out = frame.copy()
        if config.get("normalize_brightness", True):
            out = self.normalize_brightness(out, config.get("target_brightness", 100))
        if config.get("enhance_contrast", True):
            out = self.enhance_contrast(out, config.get("clahe_clip", 2.0))
        if config.get("denoise", True):
            out = self.denoise(out, config.get("denoise_strength", 8))
        if config.get("sharpen", False):
            out = self.sharpen(out)
        return out

# -------------------- Voice (Colab) --------------------
class VoiceEngine:
    """Print + optional gTTS instead of pyttsx3."""

    def __init__(self, use_gtts=True):
        self.use_gtts = use_gtts
        self.last_speech_time = 0
        self.global_cooldown = CFG.voice.global_cooldown
        self.object_cooldown_duration = CFG.voice.object_cooldown_duration
        self.object_cooldowns = defaultdict(float)
        self.log = []
        self.total = 0
        self.skipped = 0

    def announce(self, text, object_key=None, play_audio=False):
        now = time.time()
        if now - self.last_speech_time < self.global_cooldown:
            self.skipped += 1
            return False
        if object_key and now - self.object_cooldowns[object_key] < self.object_cooldown_duration:
            self.skipped += 1
            return False

        self.last_speech_time = now
        if object_key:
            self.object_cooldowns[object_key] = now
        self.total += 1
        self.log.append(text)
        print(f"[VOICE] {text}")

        if play_audio and self.use_gtts:
            try:
                from gtts import gTTS
                path = str(LOGS / "last_announce.mp3")
                gTTS(text=text, lang="en").save(path)
                display(Audio(path, autoplay=True))
            except Exception as e:
                print("[VOICE] gTTS failed:", e)
        return True

    def announce_priority(self, text, object_key=None, play_audio=False):
        return self.announce(text, object_key, play_audio)

    def get_stats(self):
        return {"total": self.total, "skipped": self.skipped, "last": self.log[-5:]}

# -------------------- Object detection --------------------
class ObjectDetector:
    def __init__(self, model_path="yolo11n.pt", conf_threshold=0.5, nms_threshold=0.45):
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[ObjectDetector] Loading {model_path} on {self.device}…")
        self.model = YOLO(model_path)
        self.model.to(self.device)
        self.track_history = defaultdict(lambda: deque(maxlen=30))
        self.inference_time = 0
        self.total_detections = 0
        self.frame_count = 0

    def detect(self, frame, conf_threshold=None, apply_tracking=True):
        conf = conf_threshold if conf_threshold is not None else self.conf_threshold
        t0 = time.time()
        results = self.model(frame, conf=conf, iou=self.nms_threshold, verbose=False)
        self.inference_time = time.time() - t0

        detections = []
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                confidence = float(box.conf[0].cpu().numpy())
                class_id = int(box.cls[0].cpu().numpy())
                class_name = result.names[class_id]
                w, h = x2 - x1, y2 - y1
                area = w * h
                if w < 10 or h < 10 or area < 400:
                    continue
                det = {
                    "class": class_name,
                    "confidence": confidence,
                    "bbox": [x1, y1, x2, y2],
                    "center": [(x1 + x2) / 2, (y1 + y2) / 2],
                    "area": area,
                    "tracked_id": None,
                    "stable": False,
                }
                detections.append(det)
                self.total_detections += 1

        if apply_tracking:
            detections = self._apply_tracking(detections)

        self.frame_count += 1
        return detections

    def _apply_tracking(self, detections):
        for det in detections:
            best_match, best_dist = None, float("inf")
            for prev_id, history in self.track_history.items():
                if not history:
                    continue
                pc = history[-1]["center"]
                cc = det["center"]
                dist = np.hypot(pc[0] - cc[0], pc[1] - cc[1])
                if dist < best_dist and dist < 50:
                    best_dist, best_match = dist, prev_id
            if best_match is not None:
                det["tracked_id"] = best_match
                self.track_history[best_match].append(det)
                det["stable"] = len(self.track_history[best_match]) > 3
            else:
                new_id = max(self.track_history.keys()) + 1 if self.track_history else 0
                det["tracked_id"] = new_id
                self.track_history[new_id].append(det)
                det["stable"] = False
        return detections

    def draw_detections(self, frame, detections, show_confidence=True):
        out = frame.copy()
        for det in detections:
            x1, y1, x2, y2 = det["bbox"]
            conf = det["confidence"]
            color = (0, 255, 0) if conf > 0.8 else (0, 255, 255) if conf > 0.6 else (0, 165, 255)
            label = det["class"] + (f" {conf:.0%}" if show_confidence else "")
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            tw = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0][0]
            cv2.rectangle(out, (x1, y1 - 25), (x1 + tw + 5, y1), color, -1)
            cv2.putText(out, label, (x1 + 2, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        return out

    def get_stats(self):
        return {
            "total_detections": self.total_detections,
            "frames": self.frame_count,
            "inference_ms": round(self.inference_time * 1000, 2),
        }

# -------------------- Currency detection --------------------
class CurrencyDetector:
    DENOMINATIONS = [10, 20, 50, 100, 200, 500]
    COLOR_RANGES = {
        10: ([5, 50, 100], [15, 255, 255]),
        20: ([140, 50, 100], [160, 255, 255]),
        50: ([100, 30, 100], [130, 255, 255]),
        100: ([10, 100, 100], [25, 255, 255]),
        200: ([20, 100, 100], [35, 255, 255]),
        500: ([150, 40, 100], [180, 255, 255]),  # pink/magenta (OpenCV H is 0-179)
    }

    def __init__(self, model_path="weights/best.pt", use_custom_model=True):
        self.model = None
        self.coco_model = None
        self.is_custom = False
        self.detected_currencies = defaultdict(float)
        self.detection_cooldown = CFG.currency_detection.detection_cooldown
        self.inference_time = 0
        self.detections_count = 0
        self.frame_count = 0

        if use_custom_model and os.path.exists(model_path):
            try:
                self.model = YOLO(model_path)
                self.is_custom = True
                print(f"[CurrencyDetector] Custom model: {model_path}")
            except Exception as e:
                print("[CurrencyDetector] Custom load failed:", e)

        if not self.is_custom:
            self.coco_model = YOLO("yolo11n.pt")
            print("[CurrencyDetector] Using YOLO + color/aspect fallback")

    def _enhance_low_light(self, frame):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    def _parse_denom(self, class_name):
        name = class_name.lower()
        for d in self.DENOMINATIONS:
            if str(d) in name:
                return d
        return None

    def _classify_by_color(self, roi):
        if roi.size == 0:
            return None
        best, score = None, 0
        for denom, (lo, hi) in self.COLOR_RANGES.items():
            mask = cv2.inRange(roi, np.array(lo), np.array(hi))
            s = cv2.countNonZero(mask) / (roi.shape[0] * roi.shape[1] + 1e-6)
            if s > score:
                best, score = denom, s
        return best if score > 0.15 else None

    def detect(self, frame, conf_threshold=0.6, apply_low_light_enhancement=True):
        if apply_low_light_enhancement:
            frame = self._enhance_low_light(frame)

        t0 = time.time()
        detections = []

        if self.is_custom and self.model:
            results = self.model(frame, conf=conf_threshold, verbose=False)
            for result in results:
                for box in result.boxes:
                    conf = float(box.conf[0].cpu().numpy())
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    denom = self._parse_denom(result.names[int(box.cls[0])])
                    if denom:
                        detections.append({
                            "denomination": denom,
                            "confidence": conf,
                            "bbox": [x1, y1, x2, y2],
                            "center": [(x1 + x2) / 2, (y1 + y2) / 2],
                            "area": (x2 - x1) * (y2 - y1),
                            "is_new": True,
                        })
        else:
            results = self.coco_model(frame, conf=conf_threshold, verbose=False)
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    w, h = x2 - x1, y2 - y1
                    ar = w / (h + 1e-3)
                    if 1.8 < ar < 3.0:
                        denom = self._classify_by_color(hsv[y1:y2, x1:x2])
                        if denom:
                            detections.append({
                                "denomination": denom,
                                "confidence": 0.7,
                                "bbox": [x1, y1, x2, y2],
                                "center": [(x1 + x2) / 2, (y1 + y2) / 2],
                                "area": w * h,
                                "is_new": True,
                            })

        self.inference_time = time.time() - t0
        now = time.time()
        for det in detections:
            d = det["denomination"]
            det["is_new"] = now - self.detected_currencies[d] > self.detection_cooldown
            if det["is_new"]:
                self.detected_currencies[d] = now

        self.detections_count += len(detections)
        self.frame_count += 1
        return detections

    def draw_detections(self, frame, detections, show_confidence=True):
        out = frame.copy()
        for det in detections:
            x1, y1, x2, y2 = det["bbox"]
            conf = det["confidence"]
            color = (0, 255, 0) if conf > 0.7 else (0, 255, 255)
            label = f"₹{det['denomination']}" + (f" {conf:.0%}" if show_confidence else "")
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            tw = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)[0][0]
            cv2.rectangle(out, (x1, y1 - 30), (x1 + tw + 10, y1), color, -1)
            cv2.putText(out, label, (x1 + 5, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        return out

    def get_stats(self):
        return {
            "total": self.detections_count,
            "frames": self.frame_count,
            "inference_ms": round(self.inference_time * 1000, 2),
            "model": "custom" if self.is_custom else "fallback",
        }

print("Core modules ready.")

## 5. Initialize detectors

In [ ]:
preprocessor = FramePreprocessor()
voice = VoiceEngine(use_gtts=CFG.voice.use_gtts)

object_detector = ObjectDetector(
    model_path=CFG.object_detection.model_path,
    conf_threshold=CFG.object_detection.conf_threshold,
    nms_threshold=CFG.object_detection.nms_threshold,
)

currency_detector = CurrencyDetector(
    model_path=CFG.currency_detection.model_path,
    use_custom_model=CFG.currency_detection.use_custom_model,
)

def run_detection(frame_bgr, mode="object", conf=None, announce=True, play_audio=False):
    """Run one frame through the selected pipeline. Returns (annotated_bgr, summary_str)."""
    cfg = asdict(CFG.preprocessing)
    frame = preprocessor.preprocess_for_detection(frame_bgr, cfg)
    lines = []

    if mode == "object":
        c = conf if conf is not None else CFG.object_detection.conf_threshold
        dets = object_detector.detect(frame, conf_threshold=c)
        annotated = object_detector.draw_detections(frame, dets)
        for d in dets:
            msg = f"{d['class']} ({d['confidence']:.0%})"
            lines.append(msg)
            if announce and d.get("stable", True):
                voice.announce(f"{d['class']} detected", object_key=d["class"], play_audio=play_audio)
        stats = object_detector.get_stats()
    elif mode == "currency":
        c = conf if conf is not None else CFG.currency_detection.conf_threshold
        dets = currency_detector.detect(frame, conf_threshold=c)
        annotated = currency_detector.draw_detections(frame, dets)
        for d in dets:
            msg = f"₹{d['denomination']} ({d['confidence']:.0%})"
            lines.append(msg)
            if announce and d.get("is_new", True):
                voice.announce_priority(f"Detected rupee {d['denomination']}", object_key=f"c_{d['denomination']}", play_audio=play_audio)
        stats = currency_detector.get_stats()
    else:
        annotated = frame
        stats = {}

    summary = "\n".join(lines) if lines else "No detections"
    summary += f"\n\nstats: {stats}"
    return annotated, summary

print("Detectors initialized.")
print("Object:", object_detector.get_stats())
print("Currency:", currency_detector.get_stats())

## 6. Quick test — single image
Upload an image, or use a sample URL.

In [ ]:
import urllib.request
from matplotlib import pyplot as plt

# Option A: download a sample photo
sample_path = SAMPLES / "sample.jpg"
if not sample_path.exists():
    url = "https://ultralytics.com/images/bus.jpg"
    urllib.request.urlretrieve(url, sample_path)
    print("Downloaded", sample_path)

# Option B: upload your own (uncomment)
# from google.colab import files
# up = files.upload()
# sample_path = SAMPLES / list(up.keys())[0]
# open(sample_path, "wb").write(up[list(up.keys())[0]])

img = cv2.imread(str(sample_path))
assert img is not None, "Failed to load image"

annotated, summary = run_detection(img, mode="object", announce=True, play_audio=False)
print(summary)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Object detection")
plt.show()

## 7. Process a video file
Writes an annotated MP4 and shows a few sample frames.

In [ ]:
def process_video(input_path, output_path=None, mode="object", conf=None, max_frames=None, skip=2):
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 20
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_path = Path(output_path or (SAMPLES / f"out_{mode}.mp4"))

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        max(fps / max(skip, 1), 1),
        (w, h),
    )

    idx, written = 0, 0
    previews = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        idx += 1
        if idx % skip != 0:
            continue
        annotated, _ = run_detection(frame, mode=mode, conf=conf, announce=False)
        # resize annotation back if preprocess resized — here preprocess keeps size
        if annotated.shape[1] != w or annotated.shape[0] != h:
            annotated = cv2.resize(annotated, (w, h))
        writer.write(annotated)
        written += 1
        if len(previews) < 3:
            previews.append(annotated)
        if max_frames and written >= max_frames:
            break

    cap.release()
    writer.release()
    print(f"Wrote {written} frames → {output_path}")
    return output_path, previews

# Example: uncomment after uploading a video to SAMPLES
# from google.colab import files
# up = files.upload()
# vid = SAMPLES / list(up.keys())[0]
# open(vid, "wb").write(up[list(up.keys())[0]])
# out, previews = process_video(vid, mode="object", max_frames=60)
# for p in previews:
#     plt.figure(figsize=(10, 6)); plt.imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

print("process_video() ready. Upload a video and uncomment the example block.")

## 8. Interactive Gradio app (webcam / upload)

Use **webcam** or **upload** an image. Switch mode between object and currency.

> Tip: for best currency results, upload `best.pt` in section 2. Without it, color/aspect fallback is used.

In [ ]:
import gradio as gr

def gradio_infer(image, mode, conf, speak):
    if image is None:
        return None, "Provide an image or enable webcam."
    # Gradio gives RGB uint8
    frame = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    # scale for speed
    scale = CFG.camera.resize_scale
    if scale < 1.0:
        h, w = frame.shape[:2]
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))
    annotated, summary = run_detection(
        frame,
        mode=mode,
        conf=float(conf),
        announce=True,
        play_audio=bool(speak),
    )
    return cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), summary

demo = gr.Interface(
    fn=gradio_infer,
    inputs=[
        gr.Image(sources=["webcam", "upload"], type="numpy", label="Camera / Image"),
        gr.Radio(["object", "currency"], value="object", label="Mode"),
        gr.Slider(0.1, 0.95, value=0.5, step=0.05, label="Confidence"),
        gr.Checkbox(value=False, label="Speak with gTTS (slower)"),
    ],
    outputs=[
        gr.Image(type="numpy", label="Detections"),
        gr.Textbox(label="Summary / announcements"),
    ],
    title="Vision AI — Object & Currency Detection",
    description="Colab port of the Flask Vision AI project. GPU recommended.",
    flagging_mode="never",
)

demo.launch(share=True, debug=False)

## 9. Optional — live webcam loop (Colab JS camera)

Captures one snapshot from the browser camera and runs detection. Re-run the cell for another frame.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="photo.jpg", quality=0.8):
    js = Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = 'Capture';
        div.appendChild(capture);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }
    ''')
    display(js)
    data = eval_js(f"takePhoto({quality})")
    binary = b64decode(data.split(",")[1])
    path = SAMPLES / filename
    with open(path, "wb") as f:
        f.write(binary)
    return path

# Uncomment to use:
# photo = take_photo()
# frame = cv2.imread(str(photo))
# annotated, summary = run_detection(frame, mode="object", play_audio=False)
# print(summary)
# plt.figure(figsize=(10, 7)); plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

print("take_photo() ready — uncomment the block above to capture from webcam.")

## How to run (checklist)

1. **Runtime → GPU**
2. Run cells **1 → 5** (install, folders, config, modules, init)
3. Run **§6** for a quick object-detection smoke test
4. Run **§8** for the Gradio webcam/upload UI
5. (Optional) Set `UPLOAD_CURRENCY_MODEL = True` in §2 and upload `best.pt` from this repo’s `weights/` folder

### Desktop vs Colab

| Feature | Desktop (`python r.py`) | This notebook |
|---|---|---|
| Continuous webcam stream | Flask MJPEG | Gradio / snapshot |
| Voice | pyttsx3 | Print + optional gTTS |
| Object YOLO | `weights/yolo11n.pt` | auto `yolo11n.pt` |
| Currency model | `weights/best.pt` | upload or color fallback |
| Keyboard / browser UI | `templates/index.html` | Gradio controls |